In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [60]:
df = pd.read_csv('hospital_patients_real_world.csv')

### 1) Understanding the dataset

In [96]:
print("Dataset Info: \n")
df.info()

Dataset Info: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   PatientID      5000 non-null   object        
 1   Age            5000 non-null   float64       
 2   Gender         5000 non-null   object        
 3   Diagnosis      5000 non-null   object        
 4   AdmissionDate  5000 non-null   datetime64[ns]
 5   DischargeDate  5000 non-null   datetime64[ns]
 6   HospitalID     5000 non-null   object        
 7   LengthOfStay   5000 non-null   int64         
 8   AdmissionYear  5000 non-null   int32         
 9   StayType       5000 non-null   object        
dtypes: datetime64[ns](2), float64(1), int32(1), int64(1), object(5)
memory usage: 371.2+ KB


In [62]:
print("\nSummary of Numerical Columns:")
display(df.describe())


Summary of Numerical Columns:


,Age
count,4650.000000
mean,47.384301
std,27.880535
min,0.000000
25%,23.000000
50%,47.000000
75%,72.000000
max,95.000000


In [63]:
print("\nFirst 5 Rows: ")
display(df.head())


First 5 Rows: 


,PatientID,Age,Gender,Diagnosis,AdmissionDate,DischargeDate,HospitalID
0,PN-2021066,7.0,Other,Myocardial Infarction,2024-03-23,2024-03-29,HOSP-65
1,PN-4606019,36.0,Other,Pneumonia,2024-08-01,2024-08-07,HOSP-79
2,PN-2594016,70.0,Other,Influenza,2024-11-16,2024-11-23,HOSP-27
3,PN-6906914,90.0,Unknown,Acute Bronchitis,2025-07-05,2025-07-10,HOSP-64
4,PN-4656204,0.0,Female,Type 2 Diabetes,2023-08-30,2023-08-31,HOSP-31


### 2) Detecting Anomalies

In [64]:
missing = df.isnull().sum()
print("Missing Values per Column:\n\n",missing)

Missing Values per Column:

 PatientID          0
Age              350
Gender           350
Diagnosis        350
AdmissionDate      0
DischargeDate      0
HospitalID         0
dtype: int64


In [ ]:
df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'])
df['DischargeDate'] = pd.to_datetime(df['DischargeDate'])



date_errors = df[df['DischargeDate'] < df['AdmissionDate']]
print(f"\nNumber of Date Logical Date Errors found: {len(date_errors)}")


Number of Date Logical Date Errors found: 150


In [ ]:
age_outliers = df[(df['Age'] < 0) | (df['Age'] > 100)]
print(f"Number of Age Outliers: {len(age_outliers)}")

Number of Age Outliers: 0


### 3) Data Cleaning

In [ ]:
mask = df['DischargeDate'] < df['AdmissionDate']
df.loc[mask, ['AdmissionDate', 'DischargeDate']] = df.loc[mask, ['DischargeDate', 'AdmissionDate']].values

print("Date errors fixed by swapping Admission and Discharge columns.")

Date errors fixed by swapping Admission and Discharge columns.


In [68]:
df['Diagnosis'] = df['Diagnosis'].fillna('Unknown')
df['Gender'] = df['Gender'].fillna('Unknown')
df['Age'] = df['Age'].fillna(df['Age'].median())

In [ ]:
print(f"Missing values remaining: {df.isnull().sum().sum()}")

Missing values remaining: 0


### 4) Descriptive Analysis

In [70]:
mean_age = df['Age'].mean()
median_age = df['Age'].median()
print(f"Mean Age: {mean_age:.2f}, \nMedian Age: {median_age}")

Mean Age: 47.36, 
Median Age: 47.0


In [71]:
print("Gender Breakdown: \n")
gender_counts = df['Gender'].value_counts(dropna=False)
print(gender_counts)

Gender Breakdown: 

Gender
Unknown    1513
Other      1223
Female     1153
Male       1111
Name: count, dtype: int64


In [72]:
total_hospitals = df['HospitalID'].nunique()

total_patients = df['PatientID'].nunique()

total_admissions = len(df)


print(f"Total Unique Hospitals: {total_hospitals}")
print(f"Total Unique Patients: {total_patients}")
print(f"Total Admissions: {total_admissions}")

Total Unique Hospitals: 90
Total Unique Patients: 5000
Total Admissions: 5000


In [73]:
df['LengthOfStay'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days

avg_los = df['LengthOfStay'].mean()
max_los = df['LengthOfStay'].max()
min_los = df['LengthOfStay'].min()

print(f"Average Length of Stay: {avg_los:.2f} days")
print(f"Stay Range: {min_los} to {max_los} days")

Average Length of Stay: 5.45 days
Stay Range: 1 to 10 days


In [74]:
min_age = df['Age'].min()
max_age = df['Age'].max()
age_range = max_age - min_age

std_age = df['Age'].std()

print(f"Age Dispersion Metrics: \n")
print(f"Youngest Patient: {min_age} years old")
print(f"Oldest Patient: {max_age} years old")
print(f"Total Age Range: {age_range} years")
print(f"Standard Deviation of Age: {std_age:.2f}")

Age Dispersion Metrics: 

Youngest Patient: 0.0 years old
Oldest Patient: 95.0 years old
Total Age Range: 95.0 years
Standard Deviation of Age: 26.89


In [75]:
df['AdmissionYear'] = df['AdmissionDate'].dt.year

yearly_counts = df['AdmissionYear'].value_counts().sort_index()

print("Admissions by Year: \n")
print(yearly_counts)

Admissions by Year: 

AdmissionYear
2023    1184
2024    1856
2025    1775
2026     185
Name: count, dtype: int64
